# BART Large MNLI – Zero-Shot Text Classification

**Model:** `facebook/bart-large-mnli`  
**Task:** Zero-shot classification via Natural Language Inference (NLI)  
**Architecture:** BART-large fine-tuned on MultiNLI  
**License:** MIT  
**Marketplace price:** \$0.10/hr

This notebook shows how to:
1. Deploy the model endpoint from AWS Marketplace
2. Classify text against arbitrary candidate labels (no training required)
3. Enable multi-label classification with `multi_label=True`
4. Apply zero-shot classification to a **ticket routing** use case

## Prerequisites

- AWS account with SageMaker execution role that has `AmazonSageMakerFullAccess`
- The model must be subscribed to in AWS Marketplace before deploying
- `boto3` and `sagemaker` Python packages installed

In [ ]:
import boto3
import sagemaker
import json
import time

sess = sagemaker.Session()
role = sagemaker.get_execution_role()
region = sess.boto_region_name

print(f"Region : {region}")
print(f"Role   : {role}")

## 1. Deploy the Endpoint

Retrieve the model package ARN from your AWS Marketplace subscription and deploy
it as a real-time SageMaker endpoint. BART-large requires `ml.g4dn.xlarge` for
GPU-accelerated inference at the \$0.10/hr rate.

In [ ]:
# Replace with the model package ARN from your AWS Marketplace subscription
MODEL_PACKAGE_ARN = "<YOUR_MODEL_PACKAGE_ARN>"

ENDPOINT_NAME = "bart-large-mnli-endpoint"
INSTANCE_TYPE = "ml.g4dn.xlarge"
INSTANCE_COUNT = 1

sm_client = boto3.client("sagemaker", region_name=region)

model_name = f"bart-large-mnli-{int(time.time())}"
sm_client.create_model(
    ModelName=model_name,
    ExecutionRoleArn=role,
    PrimaryContainer={"ModelPackageName": MODEL_PACKAGE_ARN},
)

config_name = f"{model_name}-config"
sm_client.create_endpoint_config(
    EndpointConfigName=config_name,
    ProductionVariants=[
        {
            "VariantName": "AllTraffic",
            "ModelName": model_name,
            "InstanceType": INSTANCE_TYPE,
            "InitialInstanceCount": INSTANCE_COUNT,
        }
    ],
)

sm_client.create_endpoint(
    EndpointName=ENDPOINT_NAME,
    EndpointConfigName=config_name,
)

print(f"Endpoint '{ENDPOINT_NAME}' creation started. Waiting for InService state...")

In [ ]:
waiter = sm_client.get_waiter("endpoint_in_service")
waiter.wait(
    EndpointName=ENDPOINT_NAME,
    WaiterConfig={"Delay": 30, "MaxAttempts": 30},
)
print(f"Endpoint '{ENDPOINT_NAME}' is InService and ready for inference.")

## 2. Zero-Shot Classification – Single Label

The model accepts:
- `text` – the document to classify
- `labels` – a list of candidate class names (arbitrary; no training required)

It returns each label with a probability score. The highest-scoring label is the
predicted class.

In [ ]:
runtime = boto3.client("sagemaker-runtime", region_name=region)

def classify(text: str, labels: list, multi_label: bool = False,
             endpoint_name: str = ENDPOINT_NAME) -> dict:
    """Run zero-shot classification on `text` against `labels`."""
    payload = json.dumps(
        {
            "text": text,
            "labels": labels,
            "multi_label": multi_label,
        }
    )
    response = runtime.invoke_endpoint(
        EndpointName=endpoint_name,
        ContentType="application/json",
        Body=payload,
    )
    return json.loads(response["Body"].read().decode("utf-8"))


def display_classification(result: dict) -> None:
    """Pretty-print classification scores sorted by confidence."""
    pairs = sorted(
        zip(result["labels"], result["scores"]),
        key=lambda x: x[1],
        reverse=True,
    )
    print(f"Sequence: {result['sequence']!r}\n")
    print(f"{'Label':<25} {'Score':>7}")
    print("-" * 35)
    for label, score in pairs:
        bar = "█" * int(score * 30)
        print(f"{label:<25} {score:>7.4f}  {bar}")


# Basic single-label example
result = classify(
    text="Scientists discover new exoplanet in the habitable zone of a distant star.",
    labels=["science", "politics", "sports", "entertainment", "technology"],
)
display_classification(result)

## 3. Multi-Label Classification with `multi_label=True`

By default the model treats classification as **single-label** (scores sum to 1
after a softmax over entailment vs. contradiction logits). Setting
`multi_label=True` runs independent entailment scoring for each label, allowing
multiple labels to score highly simultaneously.

Use this when a document can belong to more than one category.

In [ ]:
# A news headline that is both about technology AND business
multi_label_text = (
    "Amazon announces record quarterly earnings driven by AWS cloud growth "
    "and new AI product launches."
)

result_single = classify(
    text=multi_label_text,
    labels=["technology", "business", "politics", "health", "sports"],
    multi_label=False,
)
print("--- Single-label mode ---")
display_classification(result_single)

print()

result_multi = classify(
    text=multi_label_text,
    labels=["technology", "business", "politics", "health", "sports"],
    multi_label=True,
)
print("--- Multi-label mode (multi_label=True) ---")
display_classification(result_multi)

## 4. Use Case – Automatic Ticket Routing

Zero-shot classification excels at **routing support tickets** to the correct
team without any labelled training data. Define your routing categories as
candidate labels and classify each incoming ticket.

This allows you to change routing categories at any time by simply updating
the label list – no retraining required.

In [ ]:
# Support ticket routing labels
ROUTING_LABELS = [
    "billing and payment",
    "account access and login",
    "technical bug or error",
    "feature request",
    "data privacy and security",
    "general inquiry",
]

tickets = [
    "I was charged twice for my subscription this month and need a refund.",
    "I can't log in to my account. The password reset email is not arriving.",
    "The export button on the dashboard crashes my browser every time I click it.",
    "It would be great if you could add a dark mode to the mobile app.",
    "I'm concerned about where my personal data is stored and who can access it.",
]

print(f"{'Ticket (truncated)':<55} {'Routed to':<30} {'Confidence':>10}")
print("=" * 100)

for ticket in tickets:
    result = classify(text=ticket, labels=ROUTING_LABELS)
    top_label = result["labels"][0]
    top_score = result["scores"][0]
    truncated = ticket[:52] + "..." if len(ticket) > 55 else ticket
    print(f"{truncated:<55} {top_label:<30} {top_score:>10.4f}")

## 5. Escalation: Multi-Label Ticket Routing

Some tickets touch multiple teams. Using `multi_label=True` lets you route a
ticket to all relevant teams when confidence exceeds a threshold.

In [ ]:
complex_ticket = (
    "I was charged incorrectly AND now I can't access my account at all. "
    "Please help urgently."
)

THRESHOLD = 0.5

result_multi = classify(
    text=complex_ticket,
    labels=ROUTING_LABELS,
    multi_label=True,
)

routed_teams = [
    (label, score)
    for label, score in zip(result_multi["labels"], result_multi["scores"])
    if score >= THRESHOLD
]

print(f"Ticket: {complex_ticket!r}\n")
if routed_teams:
    print(f"Teams notified (score ≥ {THRESHOLD}):")
    for label, score in routed_teams:
        print(f"  • {label}: {score:.4f}")
else:
    print("No team scored above threshold; routing to general inquiry.")

## 6. Clean Up – Delete the Endpoint

Delete the endpoint when you are done to avoid ongoing charges (\$0.10/hr).

In [ ]:
sm_client.delete_endpoint(EndpointName=ENDPOINT_NAME)
sm_client.delete_endpoint_config(EndpointConfigName=config_name)
sm_client.delete_model(ModelName=model_name)
print(f"Endpoint '{ENDPOINT_NAME}' and associated resources deleted.")